# Modelo de Clasificación XGBoot
## Comparación de Clasificadores XGBoost para Mecanismos de Suicidio

Implementación del odelo de clasificación XGBoost para predecir el mecanismo causal de suicidio basado en diferentes variables predictoras.

In [47]:
# Importar bibliotecas necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [48]:
# Cargar y preparar los datos
try:
    # Cargar los datos directamente desde GitHub
    url = "https://raw.githubusercontent.com/jthowinsson/Suicidio_Colombia/main/Presuntos_Suicidios_con_Coor.csv"
    data = pd.read_csv(url, encoding="utf-8")
    print("Archivo cargado exitosamente")
    print("Dimensiones del dataset:", data.shape)
    
    # Función para limpiar nombres de columnas
    def limpiar_nombres(cols):
        cols = cols.astype(str)
        # Quitar tildes
        mapping_tildes = str.maketrans("áéíóúÁÉÍÓÚ", "aeiouAEIOU")
        cols = cols.str.translate(mapping_tildes)
        # A minúsculas
        cols = cols.str.lower()
        # Reemplazar cualquier carácter no alfanumérico por guion bajo
        cols = cols.str.replace(r'[^a-zA-Z0-9]+', '_', regex=True)
        # Limpiar guiones bajos extra al inicio/final
        cols = cols.str.strip('_')
        return cols

    # Aplicar limpieza a los nombres de columnas
    data.columns = limpiar_nombres(data.columns)
    print("\nColumnas después de la limpieza:")
    print(data.columns.tolist())
    
    # Verificar y limpiar la columna mecanismo_causal
    if 'mecanismo_causal' in data.columns:
        # Convertir a mayúsculas y eliminar espacios
        data['mecanismo_causal'] = data['mecanismo_causal'].str.upper().str.strip()
        print("\nValores únicos en mecanismo_causal:")
        print(data['mecanismo_causal'].value_counts())
        
        # Preparar la variable objetivo
        data['target'] = (data['mecanismo_causal'] == 'GENERADORES DE ASFIXIA').astype(int)
        print("\nDistribución de target:")
        print(data['target'].value_counts(normalize=True))
    else:
        print("\nColumnas disponibles:", data.columns.tolist())
        raise KeyError("La columna 'mecanismo_causal' no se encuentra en el dataset")

    # Definir características (usando nombres limpios)
    categorical_features = ['departamento_del_hecho_dane', 'municipio_del_hecho_dane', 
                          'zona_del_hecho', 'sexo_de_la_victima', 'estado_civil', 
                          'escolaridad', 'manera_de_muerte', 'escenario_del_hecho', 
                          'actividad_durante_el_hecho', 'edad_judicial',
                          'mes_del_hecho', 'dia_del_hecho', 'rango_de_hora_del_hecho_x_3_horas']
    
    numerical_features = ['a_o_del_hecho', 'latitud', 'longitud']

    # Verificar la existencia de todas las columnas
    all_features = categorical_features + numerical_features
    missing_cols = [col for col in all_features if col not in data.columns]
    
    if missing_cols:
        print("\nNombres de columnas actuales:", data.columns.tolist())
        raise KeyError(f"Columnas faltantes en el dataset: {missing_cols}")
    
    # Dividir los datos
    X = data[all_features]
    y = data['target']
    
    # Verificar valores nulos
    print("\nValores nulos por columna:")
    print(X.isnull().sum())
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Crear el pipeline de preprocesamiento
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])

    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    
    print("\nDatos preparados exitosamente")
    print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
    print(f"Tamaño del conjunto de prueba: {X_test.shape}")

except Exception as e:
    print("Error:", str(e))
    import traceback
    traceback.print_exc()

Archivo cargado exitosamente
Dimensiones del dataset: (23544, 42)

Columnas después de la limpieza:
['id', 'a_o_del_hecho', 'grupo_de_edad_de_la_victima', 'grupo_mayor_menor_de_edad', 'edad_judicial', 'ciclo_vital', 'sexo_de_la_victima', 'estado_civil', 'pais_de_nacimiento_de_la_victima', 'escolaridad', 'pertenencia_grupal', 'mes_del_hecho', 'dia_del_hecho', 'rango_de_hora_del_hecho_x_3_horas', 'codigo_dane_municipio', 'municipio_del_hecho_dane', 'departamento_del_hecho_dane', 'codigo_dane_departamento', 'escenario_del_hecho', 'zona_del_hecho', 'actividad_durante_el_hecho', 'circunstancia_del_hecho', 'manera_de_muerte', 'mecanismo_causal', 'diagnostico_topografico_de_la_lesion', 'presunto_agresor', 'condicion_de_la_victima', 'medio_de_desplazamiento_o_transporte', 'servicio_del_vehiculo', 'clase_o_tipo_de_accidente', 'objeto_de_colision', 'servicio_del_objeto_de_colision', 'razon_del_suicidio', 'localidad_del_hecho', 'ancestro_racial', 'codigo_dane_municipio_norm', 'muni_cod', 'muni_no

In [49]:
# Crear y entrenar el modelo XGBoost con manejo de errores
try:
    print("Iniciando entrenamiento del modelo...")
    
    # Verificar que las variables necesarias existen
    required_vars = ['X_train', 'X_test', 'y_train', 'y_test', 'preprocessor']
    for var in required_vars:
        if var not in globals():
            raise NameError(f"Variable '{var}' no encontrada. Asegúrate de que la celda anterior se ejecutó correctamente.")
    
    print("Shapes de los datos:")
    print(f"X_train: {X_train.shape}")
    print(f"X_test: {X_test.shape}")
    print(f"y_train: {y_train.shape}")
    print(f"y_test: {y_test.shape}")
    
    print("\nDistribución de clases en entrenamiento:")
    print(pd.Series(y_train).value_counts(normalize=True))
    
    # Crear el pipeline
    print("\nCreando pipeline con XGBoost...")
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42,
            eval_metric='logloss'     # Métrica de evaluación explícita
        ))
    ])

    # Entrenar el modelo
    print("\nEntrenando el modelo...")
    model.fit(X_train, y_train)
    print("Entrenamiento completado")

    # Realizar predicciones
    print("\nRealizando predicciones...")
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:,1]

    # Calcular métricas
    print("\nCalculando métricas de rendimiento...")
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)

    # Mostrar resultados
    print("\n=== Resultados de la Evaluación ===")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print("\nMatriz de Confusión:")
    print(conf_matrix)
    print("\nInforme de Clasificación:")
    print(class_report)
    
    # Visualizar importancia de características
    print("\nCalculando importancia de características...")
    
    # Primero aplicamos el preprocessor a los datos de entrenamiento
    X_train_transformed = model.named_steps['preprocessor'].transform(X_train)
    
    # Obtenemos los nombres de las características numéricas
    feature_names = numerical_features.copy()
    
    # Obtenemos los nombres de las características categóricas después de one-hot encoding
    categorical_encoder = model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
    
    # Obtener todas las características categóricas de una vez
    if hasattr(categorical_encoder, 'get_feature_names_out'):
        # Para scikit-learn >= 1.0
        cat_feature_names = categorical_encoder.get_feature_names_out(categorical_features)
    else:
        # Para versiones anteriores
        cat_feature_names = []
        for i, (feature, categories) in enumerate(zip(categorical_features, categorical_encoder.categories_)):
            for category in categories[1:]:  # Skip first category due to drop='first'
                cat_feature_names.append(f"{feature}_{category}")
    
    feature_names.extend(cat_feature_names)
    
    # Obtenemos las importancias
    importances = model.named_steps['classifier'].feature_importances_
    
    # Creamos un DataFrame con las importancias
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("\nTop 20 características más importantes:")
    print(importance_df.head(20).to_string(index=False))
    
    # Agregamos estadísticas por grupo de características
    feature_groups = {}
    
    # Agregamos características numéricas
    for feat in numerical_features:
        mask = importance_df['feature'] == feat
        feature_groups[feat] = importance_df[mask]['importance'].sum()
    
    # Agregamos características categóricas
    for feat in categorical_features:
        mask = importance_df['feature'].str.startswith(feat + '_')
        feature_groups[feat] = importance_df[mask]['importance'].sum()
    
    # Convertimos a DataFrame y ordenamos
    groups_df = pd.DataFrame({
        'grupo_caracteristica': list(feature_groups.keys()),
        'importancia_total': list(feature_groups.values())
    }).sort_values('importancia_total', ascending=False)
    
    print("\nImportancia por grupo de características:")
    print(groups_df.to_string(index=False))

except Exception as e:
    print("\n=== Error durante la ejecución ===")
    print(f"Tipo de error: {type(e).__name__}")
    print(f"Descripción: {str(e)}")
    print("\nDetalles completos del error:")
    import traceback
    traceback.print_exc()
    
    # Información adicional de diagnóstico
    print("\nInformación de diagnóstico:")
    print("Variables disponibles en el espacio de trabajo:")
    for var in ['X_train', 'X_test', 'y_train', 'y_test', 'preprocessor']:
        print(f"{var}: {'✓' if var in globals() else '✗'}")

Iniciando entrenamiento del modelo...
Shapes de los datos:
X_train: (18835, 16)
X_test: (4709, 16)
y_train: (18835,)
y_test: (4709,)

Distribución de clases en entrenamiento:
target
1    0.65357
0    0.34643
Name: proportion, dtype: float64

Creando pipeline con XGBoost...

Entrenando el modelo...
Entrenamiento completado

Realizando predicciones...

Calculando métricas de rendimiento...

=== Resultados de la Evaluación ===
Accuracy: 0.7152
ROC AUC: 0.7214

Matriz de Confusión:
[[ 546 1063]
 [ 278 2822]]

Informe de Clasificación:
              precision    recall  f1-score   support

           0       0.66      0.34      0.45      1609
           1       0.73      0.91      0.81      3100

    accuracy                           0.72      4709
   macro avg       0.69      0.62      0.63      4709
weighted avg       0.70      0.72      0.69      4709


Calculando importancia de características...

Top 20 características más importantes:
                                                 